# 02 — Zero-shot and few-shot baselines

**Purpose:** Evaluate the unchanged Qwen model in Urdu and English; few-shot demonstrations come ONLY from a training split.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# Kaggle/Colab project setup — NO PIP INSTALLATIONS.
# Uses the versions already installed in this runtime.
from pathlib import Path
from importlib import metadata
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PULL_LATEST = False  # Enable only for a clean clone after pushing fixes to GitHub.
if Path("/kaggle/working").is_dir():
    WORKSPACE = Path("/kaggle/working")
elif Path("/content").is_dir():
    WORKSPACE = Path("/content")
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / "nlp_research_proj"
ROOT = REPO_DIR / "Urdu_Qwen_Research_Notebooks"

if (REPO_DIR / ".git").is_dir():
    if PULL_LATEST:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git folder: {REPO_DIR}; inspect before removing it")
elif (Path.cwd() / "project_utils.py").is_file():
    ROOT = Path.cwd()
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(f"Project code missing from {ROOT}")

# Check user-confirmed Kaggle versions; NEVER uninstall or upgrade preinstalled dependencies.
EXPECTED = {
    "transformers": "5.16.1",
    "peft": "0.20.0",
    "torch": "2.11.0",  # the Kaggle build has a +cu128 suffix
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
}
versions = {}
for name, expected in EXPECTED.items():
    try:
        got = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Missing preinstalled dependency: {name}") from None
    versions[name] = got
    if got.split("+")[0] != expected:
        raise RuntimeError(
            f"This code is checked for {name}={expected}, found {got}. "
            "Do NOT run pip; use the tested Kaggle image or review the API changes."
        )
for name in ("accelerate", "datasets", "pandas", "matplotlib", "pytest"):
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Required package missing: {name}. This runtime cannot run the selected notebook.") from None
print("Using preinstalled Kaggle libraries (no installations):", versions)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
api = check_training_api()
print("Project ready:", ROOT)
print("Training API ready:", api["transformers"], api["trainer_tokenizer_key"], api["warmup_key"])


Run `01_preprocess_data` first. Zero-shot means zero labeled examples in the prompt. Few-shot uses selected **training** examples as demonstrations; these are not gradient updates. Keep greedy decoding and the same exact allowed labels throughout all experiments.

In [ ]:
from collections import Counter
BASE_MODEL = cfg()['model_id']
EVAL_SPLIT = 'validation'  # use test only after deciding hyperparameters
LANG = 'ur'
N_FEWSHOT_PER_CLASS = 1
MAX_ROWS = None  # use 20 for a smoke run, None for complete split
labels = task_labels('sib200')
rows = load_task('sib200', LANG, EVAL_SPLIT)
if MAX_ROWS is not None: rows = rows[:MAX_ROWS]
print('Rows:',len(rows),'Labels:',labels)
model, tokenizer = load_model(BASE_MODEL)
print('Model loaded.')

In [ ]:
demonstrations = examples_for('sib200', N_FEWSHOT_PER_CLASS, 42, lang=LANG)
assert all(r['split']=='train' for r in demonstrations)
print('Demonstrations from training only:',Counter(r['label'] for r in demonstrations))
for condition, demos in [('zero_shot',None),('few_shot',demonstrations)]:
    output = infer_rows(model, tokenizer, rows, labels, LANG, cfg()['max_new_tokens'], demos)
    path = ROOT/'outputs/predictions'/f'sib200_{condition}_{LANG}_{EVAL_SPLIT}.csv'
    meta = dict(task='sib200',lang=LANG,split=EVAL_SPLIT,run_id=condition,model_id=BASE_MODEL,
                dataset_sha256=dataset_sha('sib200',LANG,EVAL_SPLIT),
                subset=(MAX_ROWS is not None),n_fewshot_per_class=N_FEWSHOT_PER_CLASS if demos else 0)
    save_prediction_run(path,output,meta)
    print(condition, len(output), 'saved to',path, 'valid',sum(x['valid'] for x in output))
    print('First five:',[(r['reference'],r['prediction'],r['raw_prediction']) for r in output[:5]])

When done, release GPU memory before training. **Do not use validation performance as published final test performance.**

In [ ]:
del model
import torch
if torch.cuda.is_available(): torch.cuda.empty_cache()